# Domain Configuration Generation Demo

This demo generates reasoning game domain configurations through LLM with the following steps:

1. Set up environment

In [1]:
# Environment Setup
import json
import os
import shutil
import importlib.util

from common.configs import LLM_Arguments
from common.utils import get_llm

/home/wangxiangyu/miniconda3/envs/kumo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Configure your parameters below

In [ ]:
# User Parameter Configuration (replaces command line args)
CONFIG = {
    "LLM": {
        "load_type": "OPENAI",
        "model_name_or_path": "qwen2.5-72b-instruct",
        "api_base": "http://localhost:8000/v1",
        "api_key": "EMPTY"
    },
    "Domain": {
        "Goal": "Identify the correct chemical element present in an alloy",
        "Truths": "Chemical elements",
        "Actions": ["Spectroscopy", "mass spectrometry"]
    },
    "Paths": {
        "template_path": "./templates/config_generation_template.md",
        "example_path": "./templates/example.py.txt"
    }
}

3. Initialize necessary variables

In [3]:
# Initialize LLM
llm = get_llm(LLM_Arguments(**CONFIG["LLM"]))

# Extract domain dict
domain = CONFIG["Domain"]

# Read template
with open(CONFIG["Paths"]["template_path"]) as f:
    template = f.read()

# Read example
with open(CONFIG["Paths"]["example_path"]) as f:
    example_env = f.read()

4. Construct prompt for calling LLM to generate config data

In [4]:
# Convert actions list to a single string separated by '/'
domain['Actions'] = '/'.join(domain['Actions'])

# Construct the prompt for LLM to generate the config prompt
prompt_template = template + f'''
Please generate a prompt in the domain of this:

{domain}
'''
prompt = llm.call_llm(prompt_template)
prompt = f'''
Follow this prompt to generate the configs of this reasoning game:

{prompt}
'''
prompt += (
    "- Generate at least 30 actions and 50 truths. Each action should have its outcomes in the Outcomes dict. "
    "The action cannot rule out one truth in every state."
)
check_prompt = (
    "\nCheck this generation is ended or not. "
    "If not, outputs: <STATUS>NO_END</STATUS>. If yes, outputs: <STATUS>END</STATUS>.\n"
)

print(prompt)


Follow this prompt to generate the configs of this reasoning game:

# Prompt Template

Generate a configuration in Python for a **Chemical Elements** reasoning game. The goal of the game is to determine **Identify the correct chemical element present in an alloy** based on observed test outcomes. The configuration should follow the same format as the example.

**Requirements:**

1. **Truths**: Define a list of **Chemical Elements** for **Identify the correct chemical element present in an alloy**, such as **Iron (Fe)**, **Copper (Cu)**, **Aluminum (Al)**. 
2. **Actions**: Define a list of **Spectroscopy/mass spectrometry** for **Identify the correct chemical element present in an alloy**, such as **UV-Vis Spectroscopy**, **X-Ray Fluorescence (XRF)**, **Inductively Coupled Plasma Mass Spectrometry (ICP-MS)**. 
3. **Outcomes**: For each **Action**, specify the type of outcomes (e.g., “str” or “float”) and define possible outcome states. Each outcome state should **rule out** certain **C

5. Call LLM to generate config data

In [5]:
messages = [{"role":"user", "content":prompt}]

# Loop until the generation is complete or maximum attempts reached
times = 0
while True:
    if times >= 5:
        print('The generation is failed.')
        break
    generated_configs = llm.call_llm(messages)
    messages.append({'role': 'assistant', 'content': generated_configs})

    check_messages = messages.copy()
    check_messages.append({'role': 'user', 'content': check_prompt})

    check_end = llm.call_llm(check_messages)
    check_end = check_end.split('<STATUS>')[-1].split('</STATUS>')[0]

    if check_end == 'END':
        break
    else:
        messages.append({'role': 'user', 'content': 'Continue to generate the configs.'})
        times += 1

# Extract the full configuration from the conversation history
extract_messages = messages.copy()
extract_messages.append({
    'role': 'user',
    'content': 'Extract the full configs into python data structure based on the history messages.'
})
configs = llm.call_llm(extract_messages)
configs = configs.split('```')[1].strip()
configs = configs.split('python')[1].strip()

print(configs) # Display

Truths = [
    "Iron (Fe)", "Copper (Cu)", "Aluminum (Al)", "Nickel (Ni)", "Zinc (Zn)",
    "Tin (Sn)", "Lead (Pb)", "Silver (Ag)", "Gold (Au)", "Platinum (Pt)",
    "Chromium (Cr)", "Manganese (Mn)", "Cobalt (Co)", "Titanium (Ti)", "Vanadium (V)",
    "Molybdenum (Mo)", "Tungsten (W)", "Bismuth (Bi)", "Cadmium (Cd)", "Mercury (Hg)",
    "Selenium (Se)", "Tellurium (Te)", "Antimony (Sb)", "Arsenic (As)", "Beryllium (Be)",
    "Calcium (Ca)", "Strontium (Sr)", "Barium (Ba)", "Magnesium (Mg)", "Sodium (Na)",
    "Potassium (K)", "Lithium (Li)", "Rubidium (Rb)", "Cesium (Cs)", "Francium (Fr)",
    "Boron (B)", "Silicon (Si)", "Germanium (Ge)", "Arsenic (As)", "Antimony (Sb)",
    "Tellurium (Te)", "Iodine (I)", "Astatine (At)", "Polonium (Po)", "Thallium (Tl)"
]

Actions = [
    "UV-Vis Spectroscopy", "X-Ray Fluorescence (XRF)", "Inductively Coupled Plasma Mass Spectrometry (ICP-MS)",
    "Atomic Absorption Spectroscopy (AAS)", "Energy Dispersive X-Ray Spectroscopy (EDX)", "Neutron Activa

6. Write data to env_data folder

In [6]:
# Create environment folder based on the domain goal
goal_string = '_'.join(domain['Goal'].split(' '))
env_data_dir = f'env_data/{goal_string}'
os.makedirs(env_data_dir, exist_ok=True)

# Write the configuration to a Python file
with open(f'{env_data_dir}/configs.py', 'w') as f:
    f.write(configs)

8. Verify and revise

In [7]:
def check_truth_not_rule_out(all_truths, all_actions, ta_mapping):
    """
    Check if any truth is always ruled out for a specific action in every state.
    If such a case is found, print a message and record the action key.
    """
    keys = []
    for key, value in ta_mapping.items():
        # Check each truth to see if it is always missing in every state's outcomes
        for truth in all_truths:
            flag = True
            for state in value["states"].values():
                if truth not in state:
                    flag = False
                    break
            if flag:
                print(f'{truth} will be always ruled out for {key}')
                keys.append(key)
    assert len(keys) == 0


def validate_truths_actions_against_outcomes(all_truths, all_actions, ta_mapping):
    """
    Validate that the set of truths and actions used in the outcomes mapping
    matches the provided lists. Adjusts the lists to ensure consistency.
    """
    # Extract all truths actually used in the mapping
    used_truths = set()
    for outcome in ta_mapping.values():
        for state in outcome["states"].values():
            used_truths.update(state)
    
    # Process truths
    current_truths = set(all_truths)
    unused_truths = current_truths - used_truths
    missing_truths = used_truths - current_truths
    
    # Remove unused truths and add missing ones
    if unused_truths:
        print(f'Removing unused truths: {unused_truths}')
        current_truths -= unused_truths
        
    if missing_truths:
        print(f'Adding missing truths: {missing_truths}')
        current_truths.update(missing_truths)
    
    # Verify truth consistency
    if current_truths != used_truths:
        raise AssertionError('Truths mismatch after auto-correction. Please regenerate or delete the truths in the outcomes')
    
    # Process actions
    declared_actions = set(all_actions)
    mapped_actions = set(ta_mapping.keys())
    
    undeclared_actions = mapped_actions - declared_actions
    unused_actions = declared_actions - mapped_actions
    
    # Remove unused actions
    if unused_actions:
        print(f'Removing unused actions: {unused_actions}')
        declared_actions -= unused_actions
    
    # Verify action consistency
    if undeclared_actions:
        raise AssertionError(
            f'Actions in mapping not declared: {undeclared_actions}. '
            'Please regenerate or delete these actions in the outcomes'
        )
    
    return current_truths, declared_actions, ta_mapping

In [8]:
# Dynamically load the generated configuration module
env_file_path = f'{env_data_dir}/configs.py'
if os.path.exists(env_file_path):
    spec = importlib.util.spec_from_file_location("configs", env_file_path)
    configs_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(configs_module)
    print('Load the module dynamically')
    # Extract variables from the module
    for attr in dir(configs_module):
        if not attr.startswith("__"):
            if attr == 'Actions':
                Actions = getattr(configs_module, attr)
            elif attr == 'Outcomes':
                Outcomes = getattr(configs_module, attr)
            else:
                Truths = getattr(configs_module, attr)
else:
    print(f"File not found: {env_file_path}")


# Validate the configuration consistency
check_truth_not_rule_out(Truths, Actions, Outcomes)
new_truths, new_actions, new_outcomes = validate_truths_actions_against_outcomes(Truths, Actions, Outcomes)

# Overwrite the configuration file with the updated values
updated_config = f'''Truths = {new_truths}
Actions = {new_actions}
Outcomes = {new_outcomes}
'''
with open(env_file_path, 'w') as f:
    f.write(updated_config)
    
print(updated_config) # Display

Load the module dynamically
Truths = {'Francium (Fr)', 'Iron (Fe)', 'Barium (Ba)', 'Tungsten (W)', 'Germanium (Ge)', 'Titanium (Ti)', 'Antimony (Sb)', 'Chromium (Cr)', 'Mercury (Hg)', 'Potassium (K)', 'Thallium (Tl)', 'Silicon (Si)', 'Aluminum (Al)', 'Astatine (At)', 'Molybdenum (Mo)', 'Lithium (Li)', 'Manganese (Mn)', 'Cadmium (Cd)', 'Boron (B)', 'Polonium (Po)', 'Selenium (Se)', 'Zinc (Zn)', 'Strontium (Sr)', 'Iodine (I)', 'Tellurium (Te)', 'Lead (Pb)', 'Calcium (Ca)', 'Gold (Au)', 'Rubidium (Rb)', 'Platinum (Pt)', 'Bismuth (Bi)', 'Silver (Ag)', 'Cobalt (Co)', 'Magnesium (Mg)', 'Nickel (Ni)', 'Copper (Cu)', 'Tin (Sn)', 'Arsenic (As)', 'Vanadium (V)', 'Beryllium (Be)', 'Cesium (Cs)', 'Sodium (Na)'}
Actions = {'Transmission Electron Microscopy (TEM)', 'Thermal Analysis (TA)', 'Atomic Absorption Spectroscopy (AAS)', 'Secondary Ion Mass Spectrometry (SIMS)', 'Mössbauer Spectroscopy', 'Raman Spectroscopy', 'Time-of-Flight Secondary Ion Mass Spectrometry (ToF-SIMS)', 'Electron Microprobe A

9. Build prompt for LLM to generate python code for the config data

In [9]:
# Generate the environment file based on the example template
env_prompt = f'''
Please generate the environment file based on the following template:

```python
{example_env}
```

Here's the domain for the environment:
{domain}

You only need to revise: knowledge_book_prompt, system_prompt, "xxxEnV" in registry and class name, and "from env.data.xxx_data import Truths, Actions, Outcomes" based on the given domain.
'''

print(env_prompt) # Display


Please generate the environment file based on the following template:

```python
import os
import json
from common.registry import registry
from .base_env import BaseEnv
from env.data.chemical_data import Truths, Actions, Outcomes

@registry.register_environment("ChemicalEnv")
class ChemicalEnv(BaseEnv):
    
    def __init__(self, datapoint=None):
        self.all_truths = Truths      #
        self.all_actions = Actions    #
        self.ta_mapping = Outcomes
    
        self.env = None
        
        if datapoint is not None:
            self.reset(datapoint)

    
    def get_knowledge_book_prompt(self, truths, actions, ta_mapping):
        
        knowledge_book_prompt = \
        f"""
Please write a chemical analysis guidebook that introduces the following chemical substances and experiments in natural language according to the following information.

Chemical substances: {truths}

Experiments: {actions}

Outcomes: {ta_mapping}

""" + """
Requirements:

1. The sets defined u

10. Call LLM to generate python code for the config data

In [10]:
generated_env = llm.call_llm(env_prompt)

# Extract the Python code block from LLM response
generated_code = generated_env.split('```python')[1].split('```')[0].strip()

print(generated_code) # Dislpay

# Parse environment metadata from the generated code
env_name = generated_code.split('class ')[1].split('(')[0].strip()
env_data_file = generated_code.split('from env.data.')[1].split(' import')[0].strip()

# Save environment files
env_dir = f'env_data/{goal_string}'
os.makedirs(env_dir, exist_ok=True)

with open(f'{env_dir}/env.py', 'w') as f:
    f.write(generated_code)

metadata = {
    'env_name': env_name,
    'env_data_file_name': env_data_file,
    'goal': domain['Goal'],
    'actions': domain['Actions'],
    'truths': domain['Truths'],
}

with open(f'{env_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f)

import os
import json
from common.registry import registry
from .base_env import BaseEnv
from env.data.alloy_data import Truths, Actions, Outcomes

@registry.register_environment("AlloyEnv")
class AlloyEnv(BaseEnv):
    
    def __init__(self, datapoint=None):
        self.all_truths = Truths      # Chemical elements
        self.all_actions = Actions    # Spectroscopy/mass spectrometry
        self.ta_mapping = Outcomes
    
        self.env = None
        
        if datapoint is not None:
            self.reset(datapoint)

    
    def get_knowledge_book_prompt(self, truths, actions, ta_mapping):
        
        knowledge_book_prompt = \
        f"""
Please write a chemical analysis guidebook that introduces the following chemical elements and spectroscopy/mass spectrometry experiments in natural language according to the following information.

Chemical elements: {truths}

Experiments: {actions}

Outcomes: {ta_mapping}

""" + """
Requirements:

1. The sets defined under the state 

11. Register the domain data and python code

In [11]:
# Determine class name and copy files to proper locations
class_name = env_name[:-3]
os.makedirs(f'env/data/', exist_ok=True)
shutil.copy(f'env_data/{goal_string}/configs.py', f'env/data/{env_data_file}.py')
shutil.copy(f'env_data/{goal_string}/env.py', f'env/{class_name.lower()}_env.py')

# Register the environment in the __init__.py file in the env folder
init_file = 'env/__init__.py'
with open(init_file, 'r') as file:
    lines = file.readlines()

new_import = f'from .{class_name.lower()}_env import {class_name}Env\n'
new_env = f"    '{class_name}Env',\n"

import_index = next(i for i, line in enumerate(lines) if line.startswith('from'))
all_index = next(i for i, line in enumerate(lines) if line.startswith('__all__'))

lines.insert(import_index, new_import)
lines.insert(all_index + 2, new_env)

with open(init_file, 'w') as file:
    file.writelines(lines)